# Chuuchuu punctuality statistics at the country level

Same summary as the one `Chuuchuu_pipeline.ipynb` exports (Step 7), but aggregated **by country instead of by operator**, for France, Belgium, Italy, Switzerland, Austria, Germany, the Netherlands and Luxembourg.

**Input**: the enriched per-stop parquet files of the shared drive (`processed_data/`, see `chuuchuu_paths.py`). They are read **one month at a time**: a punctuality unit never spans two months (`journey_id` carries the service date), so the monthly files give exactly the same result as the merged yearly file, with ~1/12 of the memory (the yearly file is 81M rows) and without depending on `8_Chuuchuu_merge_monthly_summary.ipynb` having been re-run.

**Output**: `summary_stats/chuuchuu_summary_country_level_<first month>_to_<last month>.xlsx`, a `Summary` sheet (all countries) and one sheet per country.

**What a country figure means.** The unit is the one of Step 7: a train's *national segment* (`journey_id_country`), i.e. the stops it makes inside one country, scored at its closing stop (the terminus if it lies in that country, otherwise the last stop before the border). Every segment of a country counts, whoever operates the train. An international train therefore counts once in each country it runs through, so **N total train counts segments, not physical trains**, and the total of the `Summary` sheet is not a number of trains.

Unlike the operator summary, where Eurostar, Ouigo, Trenitalia France, Westbahn, Flixtrain... are kept as one *whole journey*, here those trains are **also split by country**: a country figure has to be about the network of that country, and a whole journey has no single country. Everything else (ambiguous trips excluded, cancellation rules, closing stop, delay thresholds, strict `>` comparison, partially cancelled units never counted as delayed) is copied from Step 7.

## Parameters

In [1]:
COUNTRIES = ["France", "Belgium", "Italy", "Switzerland", "Austria", "Germany", "Netherlands", "Luxembourg"]

# "monthly": the monthly enriched files, read one at a time (light on memory, recommended)
# "yearly" : the merged yearly file (8_Chuuchuu_merge_monthly_summary.ipynb), read in a single pass
#            (~81M rows before the country filter : needs several GB of RAM)
SOURCE = "monthly"

# monthly source only. None = every monthly file found on the drive; or a list such as ["2025_01", "2025_02"]
MONTHS = None

In [2]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from chuuchuu_paths import ENRICHED_DIR, output_path

pd.set_option("display.max_columns", None)

assert SOURCE in ("monthly", "yearly"), "SOURCE must be 'monthly' or 'yearly'"

if SOURCE == "monthly":
    if MONTHS is None:
        # only the monthly files (data_chuuchuu_YYYY_MM_enriched.parquet), never the merged yearly one
        monthly_pattern = re.compile(r"^data_chuuchuu_(\d{4}_\d{2})_enriched\.parquet$")
        MONTHS = sorted(m.group(1) for f in ENRICHED_DIR.iterdir() if (m := monthly_pattern.match(f.name)))
    SELECTIONS = MONTHS  # one parquet file each
    FILE_LABEL = f"{MONTHS[0]}_to_{MONTHS[-1]}"
else:
    # the merged file data_chuuchuu_<first>_to_<last>_enriched.parquet, a single selection
    yearly_pattern = re.compile(r"^data_chuuchuu_(\d{4}_\d{2}_to_\d{4}_\d{2})_enriched\.parquet$")
    yearly_files = sorted(m.group(1) for f in ENRICHED_DIR.iterdir() if (m := yearly_pattern.match(f.name)))
    assert yearly_files, f"no merged yearly parquet in {ENRICHED_DIR} : run 8_Chuuchuu_merge_monthly_summary.ipynb first"
    FILE_LABEL = yearly_files[-1]
    first, last = FILE_LABEL.split("_to_")
    MONTHS = [str(p).replace("-", "_") for p in pd.period_range(first.replace("_", "-"), last.replace("_", "-"), freq="M")]
    SELECTIONS = [FILE_LABEL]
print(f"{SOURCE} source, {len(MONTHS)} months: {MONTHS}")

monthly source, 12 months: ['2025_01', '2025_02', '2025_03', '2025_04', '2025_05', '2025_06', '2025_07', '2025_08', '2025_09', '2025_10', '2025_11', '2025_12']


## Loading one month

Only the columns Step 7 needs, only the stops located in the countries of interest (filter applied at read time), and the low-cardinality text columns, including the journey ids, read as pandas categories: a month is then ~0.4 GB in memory.

In [3]:
CATEGORY_COLUMNS = [
    "date", "routeType", "journey_type", "country", "HSR_service", "LD", "night_train", "joint_service",
    "normalized_operator", "operator_for_punctuality", "depart_terminus", "arrivalCancelled_resolved",
    "journey_id", "journey_id_country",
]
OTHER_COLUMNS = ["is_ambiguous_trip", "sort_time", "arrivalDelay"]


def load_month(month):
    return pq.read_table(
        output_path(month),
        columns=CATEGORY_COLUMNS + OTHER_COLUMNS,
        filters=[("country", "in", COUNTRIES)],
        read_dictionary=CATEGORY_COLUMNS,
    ).to_pandas()

## Building the punctuality units

Step 7 of `Chuuchuu_pipeline.ipynb` (cells "Punctuality unit" to `journeys`), run on the stops of the countries of interest and with **one unit per `journey_id_country` for every train**. Since the unit is a national segment, filtering the stops by country at read time cuts no unit in two.

Besides the Step 7 columns, `delay_measured` flags the units that arrived at their closing stop **and** have an `arrivalDelay` there. See the data quality note at the end of the notebook for why it is kept.

In [4]:
DELAY_THRESHOLDS_MIN = [3, 5, 6, 10, 15, 30, 60]
UNIT = "journey_id_country"


def most_common(stops, column):
    # a unit takes its most common value of `column` (ties broken alphabetically, so reruns agree),
    # the same rule as Step 7 for operator_for_punctuality / normalized_operator
    counts = (
        stops.dropna(subset=[column])
        .groupby([UNIT, column], observed=True).size()
        .reset_index(name="n")
    )
    counts[column] = counts[column].astype(str)
    counts = counts.sort_values([UNIT, "n", column], ascending=[True, False, True])
    return counts.drop_duplicates(UNIT).set_index(UNIT)[column]


def build_units(data):
    # a journey with a repeated stop id can't be ranked into depart / intermediate / terminus (Step 2)
    clean_data = data[~data["is_ambiguous_trip"]].copy()
    service_date = pd.to_datetime(clean_data["date"].astype(str))
    clean_data["year"] = service_date.dt.year
    clean_data["month"] = service_date.dt.month

    # cancellation: only classified when every stop of the unit has a known status
    by_unit = clean_data[UNIT]
    stop_counts = pd.DataFrame({
        "n_stops": clean_data.groupby(UNIT, observed=True).size(),
        "n_known_cancellation": clean_data["arrivalCancelled_resolved"].notna().groupby(by_unit, observed=True).sum(),
        "n_cancelled_stops": (clean_data["arrivalCancelled_resolved"] == "t").groupby(by_unit, observed=True).sum(),
    })
    reliable = stop_counts["n_known_cancellation"] == stop_counts["n_stops"]
    stop_counts["is_fully_cancelled"] = reliable & (stop_counts["n_cancelled_stops"] == stop_counts["n_stops"])
    stop_counts["is_partially_cancelled"] = (
        reliable & (stop_counts["n_cancelled_stops"] > 0) & (stop_counts["n_cancelled_stops"] < stop_counts["n_stops"])
    )

    # attributes constant across a unit's stops (journey_id == agency + routeType + routeNumber + date)
    unit_attrs = clean_data.drop_duplicates(UNIT).set_index(UNIT)[
        ["journey_id", "routeType", "year", "month", "country", "journey_type", "HSR_service", "LD", "night_train", "joint_service"]
    ]

    # the stop that closes each unit: the journey's terminus if it lies in the country, otherwise the
    # segment's last stop in time (the last stop before the border), exactly as in Step 7
    terminus_rows = clean_data[clean_data["depart_terminus"] == "terminus"].set_index(UNIT)
    ranked_stops = clean_data[(clean_data["depart_terminus"] != "unknown") & clean_data["sort_time"].notna()]
    last_segment_stops = (
        ranked_stops.sort_values("sort_time", kind="stable")
        .drop_duplicates(UNIT, keep="last")
        .set_index(UNIT)
    )
    last_segment_stops = last_segment_stops.loc[last_segment_stops.index.difference(terminus_rows.index)]
    closing_rows = pd.concat([terminus_rows, last_segment_stops])
    assert closing_rows.index.is_unique, "a unit has more than one closing stop"

    # a segment made of the journey's departure station alone has no arrival on that network to
    # measure : left out rather than counted as a train that never arrived
    departure_only_units = closing_rows.index[closing_rows["depart_terminus"] == "depart"]

    units = unit_attrs.join(stop_counts).drop(index=departure_only_units)
    units["service_operator"] = most_common(clean_data, "normalized_operator")
    units["punctuality_operator"] = most_common(clean_data, "operator_for_punctuality")

    cancelled_terminus = closing_rows["arrivalCancelled_resolved"].astype(object).map({"t": True, "f": False})  # NaN where unresolved
    units["cancelled_terminus"] = units.index.map(cancelled_terminus)
    units["terminus_arrival_delay"] = units.index.map(closing_rows["arrivalDelay"])
    units["arrived_at_terminus"] = units["cancelled_terminus"] == False
    units["delay_measured"] = units["arrived_at_terminus"] & units["terminus_arrival_delay"].notna()

    # delayed by strictly more than X min at the closing stop; cancellation takes precedence, so a
    # partially cancelled unit is never delayed (see Step 7 for the reasoning and its caveat)
    not_partially_cancelled = ~units["is_partially_cancelled"].fillna(False).astype(bool)
    for threshold in DELAY_THRESHOLDS_MIN:
        units[f"delayed_{threshold}min"] = (
            units["arrived_at_terminus"] & not_partially_cancelled & (units["terminus_arrival_delay"] > threshold * 60)
        )

    # the groupbys below need the unit id as a regular column; text keys as plain objects so that
    # groupby(dropna=False) doesn't build a cartesian product of categories
    units = units.reset_index().rename(columns={UNIT: "stat_unit_id"})
    key_columns = ["routeType", "country", "journey_type", "HSR_service", "LD", "night_train", "joint_service"]
    units[key_columns] = units[key_columns].astype(object)
    units["joint_service"] = units["joint_service"].fillna("UNKNOWN_OPERATOR")
    units["service_operator"] = units["service_operator"].fillna("UNKNOWN_OPERATOR")
    units["punctuality_operator"] = units["punctuality_operator"].fillna("UNKNOWN_OPERATOR")
    return units, len(departure_only_units)

## Aggregating

Same counts as the pipeline summary, by `year / month / country / route type / journey type / HSR / long distance / night train` (**`Summary`** sheet), and the same keys plus the operator that runs the service and the joint service for the per-country sheets (as the per-operator sheets carry `Joint service`). `service_operator` is the operator the train normally belongs to (e.g. the SNCF TGV on German tracks is `SNCF`, while it counts for DB in the operator summary), `joint_service` all the operators involved in the whole train.

`punctuality_operator` is not exported: it only serves the reconciliation with the operator summary at the end.

In [5]:
SUMMARY_KEYS = ["year", "month", "country", "routeType", "journey_type", "HSR_service", "LD", "night_train"]
COUNTRY_SHEET_KEYS = SUMMARY_KEYS + ["service_operator", "joint_service"]
RECONCILIATION_KEYS = ["year", "month", "country", "punctuality_operator"]

SUMMARY_AGGREGATIONS = dict(
    n_total_train=("stat_unit_id", "size"),
    n_cancelled_full=("is_fully_cancelled", "sum"),
    n_cancelled_partial=("is_partially_cancelled", "sum"),
    n_delay_3min=("delayed_3min", "sum"),
    n_delay_5min=("delayed_5min", "sum"),
    n_delay_6min=("delayed_6min", "sum"),
    n_delay_10min=("delayed_10min", "sum"),
    n_delay_15min=("delayed_15min", "sum"),
    n_delay_30min=("delayed_30min", "sum"),
    n_delay_60min=("delayed_60min", "sum"),
    n_arriving_terminus=("arrived_at_terminus", "sum"),
    n_arriving_terminus_delay_reported=("delay_measured", "sum"),
)


def summarize(units, keys):
    table = units.groupby(keys, dropna=False).agg(**SUMMARY_AGGREGATIONS).reset_index()
    table["n_cancelled_full_or_partial"] = table["n_cancelled_full"] + table["n_cancelled_partial"]
    assert table["n_total_train"].sum() == len(units), "grouped total doesn't match the unit count : a group key must be dropping nulls"
    return table

## Running every month

One file at a time (a month, or the whole year with `SOURCE = "yearly"`): load, build the units, aggregate, keep only the (small) aggregated tables.

In [6]:
summary_parts, country_sheet_parts, reconciliation_parts = [], [], []
unit_counts = {}

for selection in SELECTIONS:
    start = time.time()
    stops = load_month(selection)
    units, n_departure_only = build_units(stops)

    # invariants that hold by construction : a failure here is a bug in the code above
    assert units["stat_unit_id"].is_unique
    assert units["country"].isin(COUNTRIES).all()
    assert not (units["is_fully_cancelled"] & units["is_partially_cancelled"]).any(), "full and partial cancellation overlap"
    assert not (units["is_partially_cancelled"] & units[[f"delayed_{t}min" for t in DELAY_THRESHOLDS_MIN]].any(axis=1)).any(), \
        "a partially cancelled unit is counted as delayed"

    summary_parts.append(summarize(units, SUMMARY_KEYS))
    country_sheet_parts.append(summarize(units, COUNTRY_SHEET_KEYS))
    reconciliation_parts.append(summarize(units, RECONCILIATION_KEYS))
    unit_counts[selection] = len(units)

    print(f"{selection}: {len(stops):>9,} stops -> {len(units):>8,} units "
          f"({n_departure_only:,} departure-only segments excluded) in {time.time() - start:.0f}s")
    del stops, units

summary = pd.concat(summary_parts, ignore_index=True)
country_sheets = pd.concat(country_sheet_parts, ignore_index=True)
reconciliation = pd.concat(reconciliation_parts, ignore_index=True)
print(f"\n{sum(unit_counts.values()):,} punctuality units in total")

2025_01: 6,410,977 stops ->  614,790 units (10,579 departure-only segments excluded) in 14s


2025_02: 5,814,555 stops ->  566,823 units (10,003 departure-only segments excluded) in 12s


2025_03: 6,255,260 stops ->  619,306 units (11,512 departure-only segments excluded) in 14s


2025_04: 5,899,174 stops ->  574,923 units (10,460 departure-only segments excluded) in 13s


2025_05: 6,581,996 stops ->  641,306 units (11,256 departure-only segments excluded) in 14s


...

2025_08: 6,013,086 stops ->  602,539 units (11,320 departure-only segments excluded) in 13s


2025_09: 6,646,425 stops ->  639,271 units (11,690 departure-only segments excluded) in 14s


2025_10: 8,142,773 stops ->  809,113 units (16,101 departure-only segments excluded) in 17s


2025_11: 8,053,536 stops ->  803,058 units (16,758 departure-only segments excluded) in 17s


2025_12: 8,772,323 stops ->  860,839 units (17,800 departure-only segments excluded) in 19s

7,955,424 punctuality units in total


## Export to Excel

In [7]:
SUMMARY_COLUMN_NAMES = {
    "year": "Year",
    "month": "Month",
    "country": "Country",
    "service_operator": "Service operator",
    "joint_service": "Joint service",
    "routeType": "Route type",
    "journey_type": "Journey type",
    "HSR_service": "HSR service",
    "LD": "Long distance",
    "night_train": "Night train",
    "n_total_train": "N total train",
    "n_cancelled_full": "N cancelled train (full cancellation)",
    "n_cancelled_partial": "N cancelled train (partial cancellation)",
    "n_cancelled_full_or_partial": "N cancelled train (full + partial)",
    "n_delay_3min": "N train delay (> 3 min)",
    "n_delay_5min": "N train delay (> 5 min)",
    "n_delay_6min": "N train delay (> 6 min)",
    "n_delay_10min": "N train delay (> 10 min)",
    "n_delay_15min": "N train delay (> 15 min)",
    "n_delay_30min": "N train delay (> 30 min)",
    "n_delay_60min": "N train delay (> 60 min)",
    "n_arriving_terminus": "N train arriving at terminus",
    "n_arriving_terminus_delay_reported": "N train arriving at terminus (delay reported)",
}
SUMMARY_KEY_COLUMNS = ["Year", "Month", "Country", "Service operator", "Joint service", "Route type", "Journey type", "HSR service", "Long distance", "Night train"]
SUMMARY_COUNT_COLUMNS = [SUMMARY_COLUMN_NAMES[c] for c in [
    "n_total_train", "n_cancelled_full", "n_cancelled_partial", "n_cancelled_full_or_partial",
    "n_delay_3min", "n_delay_5min", "n_delay_6min", "n_delay_10min", "n_delay_15min", "n_delay_30min", "n_delay_60min",
    "n_arriving_terminus", "n_arriving_terminus_delay_reported",
]]


def format_summary(table):
    # export names and column order; "Service operator" / "Joint service" only exist on the country sheets
    table = table.rename(columns=SUMMARY_COLUMN_NAMES)
    key_columns = [column for column in SUMMARY_KEY_COLUMNS if column in table.columns]
    return table[key_columns + SUMMARY_COUNT_COLUMNS].sort_values(key_columns)


summary_export = format_summary(summary)
country_exports = {
    country: format_summary(country_sheets[country_sheets["country"] == country].drop(columns="country"))
    for country in COUNTRIES
}
summary_export.head()

,Year,Month,Country,Route type,Journey type,HSR service,Long distance,Night train,N total train,N cancelled train (full cancellation),N cancelled train (partial cancellation),N cancelled train (full + partial),N train delay (> 3 min),N train delay (> 5 min),N train delay (> 6 min),N train delay (> 10 min),N train delay (> 15 min),N train delay (> 30 min),N train delay (> 60 min),N train arriving at terminus,N train arriving at terminus (delay reported)
0,2025,1,Austria,BUS,domestic,No,No,No,17,0,0,0,0,0,0,0,0,0,0,17,0
1,2025,1,Austria,BUS,international,No,No,No,94,0,0,0,0,0,0,0,0,0,0,94,0
2,2025,1,Austria,CAT,domestic,No,No,No,734,0,0,0,0,0,0,0,0,0,0,734,0
3,2025,1,Austria,D,domestic,No,Yes,No,475,0,2,2,89,53,47,23,21,5,3,473,454
4,2025,1,Austria,D,international,No,Yes,No,396,0,0,0,68,44,30,16,12,7,1,396,386


In [8]:
SUMMARY_DIR = Path("summary_stats")
SUMMARY_DIR.mkdir(exist_ok=True)
summary_path = SUMMARY_DIR / f"chuuchuu_summary_country_level_{FILE_LABEL}.xlsx"

# the main table stays the first sheet, so anything reading the file's first sheet still gets it
with pd.ExcelWriter(summary_path) as writer:
    summary_export.to_excel(writer, sheet_name="Summary", index=False)
    for country, table in country_exports.items():
        table.to_excel(writer, sheet_name=country, index=False)
print(f"Saved to {summary_path} (sheets: Summary, {', '.join(COUNTRIES)})")

Saved to summary_stats\chuuchuu_summary_country_level_2025_01_to_2025_12.xlsx (sheets: Summary, France, Belgium, Italy, Switzerland, Austria, Germany, Netherlands, Luxembourg)


## Checks

In [9]:
N, ARR = "N total train", "N train arriving at terminus"
CANCELLED, DELAYED_15 = "N cancelled train (full + partial)", "N train delay (> 15 min)"
DELAY_COLUMNS = [c for c in SUMMARY_COUNT_COLUMNS if c.startswith("N train delay")]

# 1 : the sheets are consistent with each other (same units, only split differently)
for country, table in country_exports.items():
    expected = summary_export.loc[summary_export["Country"] == country, SUMMARY_COUNT_COLUMNS].sum()
    assert (table[SUMMARY_COUNT_COLUMNS].sum() == expected).all(), f"{country} sheet doesn't add up to its Summary rows"
assert summary_export[N].sum() == sum(unit_counts.values()), "Summary total differs from the number of units built"
print("1 ok : country sheets add up to the Summary, which adds up to the number of units")

# 2 : counts are nested the way they should be
assert (summary_export[ARR] <= summary_export[N]).all()
assert (summary_export[DELAY_COLUMNS].diff(axis=1).iloc[:, 1:] <= 0).all().all(), "delay counts must decrease as the threshold grows"
assert (summary_export[DELAY_COLUMNS[0]] <= summary_export[ARR]).all()
assert (summary_export["N train arriving at terminus (delay reported)"] <= summary_export[ARR]).all()
print("2 ok : arrivals <= units, delays decrease with the threshold and stay below arrivals")

# 3 : every month and every country is there
months_found = set(zip(summary_export["Year"], summary_export["Month"]))
assert len(months_found) == len(MONTHS), f"{len(months_found)} months in the output for {len(MONTHS)} files"
missing = set(COUNTRIES) - set(summary_export["Country"])
assert not missing, f"no data at all for {missing}"
print("3 ok : every month and every country present")

1 ok : country sheets add up to the Summary, which adds up to the number of units
2 ok : arrivals <= units, delays decrease with the threshold and stay below arrivals
3 ok : every month and every country present


### Reconciliation with the operator summary

The country figures are the same units as the pipeline summary, regrouped. So, for an incumbent, the units its own network scores in the operator summary (`Punctuality scope` = `national network`) must be found again here, as the units whose `punctuality_operator` is that operator, summed over the countries. Two small differences are expected: segments outside the 8 countries (SNCF in Spain, for instance) are not loaded here, so this notebook is lower; and a train kept as one whole journey in the operator summary (Eurostar, Westbahn...) that also has stops of an incumbent is not in that incumbent's national units there, but is here, so this notebook can be slightly higher (a few units for OEBB in 2025). The check allows 0.5% of the operator's units.

The reference is the merged operator summary of the same months (`chuuchuu_summary_monthly_merged.xlsx`, written by `8_Chuuchuu_merge_monthly_summary.ipynb`); skipped if it's absent.

In [10]:
reference_path = SUMMARY_DIR / "chuuchuu_summary_monthly_merged.xlsx"
INCUMBENT_SHEETS = ["SNCF", "DB", "NS", "OEBB", "SBB", "SNCB", "Trenitalia"]

if reference_path.exists():
    reference = pd.read_excel(reference_path, sheet_name=INCUMBENT_SHEETS)
    mine = reconciliation.groupby(["year", "month", "punctuality_operator"])[["n_total_train", "n_arriving_terminus", "n_delay_15min"]].sum()

    official_parts = []
    for operator, sheet in reference.items():
        national = sheet[sheet["Punctuality scope"] == "national network"]
        official_operator = national.groupby(["Year", "Month"])[[N, ARR, DELAYED_15]].sum()
        official_operator.index = pd.MultiIndex.from_tuples(
            [(year, month, operator) for year, month in official_operator.index], names=mine.index.names
        )
        official_operator.columns = mine.columns
        official_parts.append(official_operator)
    official = pd.concat(official_parts)

    months_wanted = {tuple(int(part) for part in month.split("_")) for month in MONTHS}
    official = official[[(year, month) in months_wanted for year, month, _ in official.index]]
    gap = official.sub(mine.reindex(official.index).fillna(0))

    by_operator = gap.groupby(level="punctuality_operator").sum().join(
        official.groupby(level="punctuality_operator")["n_total_train"].sum().rename("operator summary units")
    )
    display(by_operator)
    relative_gap = gap["n_total_train"].groupby(level="punctuality_operator").sum().abs() / official["n_total_train"].groupby(level="punctuality_operator").sum()
    assert (relative_gap < 0.005).all(), f"more than 0.5% of units differ from the operator summary: {relative_gap.to_dict()}"
else:
    print(f"{reference_path} not found : reconciliation skipped")

,n_total_train,n_arriving_terminus,n_delay_15min,operator summary units
punctuality_operator,,,,
DB,0,0,0,333911
NS,0,0,0,1902783
OEBB,-13,-13,-2,141641
SBB,0,0,0,364874
SNCB,0,0,0,1045718
SNCF,852,572,26,819358
Trenitalia,5,3,0,2970347


## Data quality notes

Inherited from the enriched parquet (see the 2025 audit of the feed), they matter for the country view:

- **Switzerland**: SBB's feed has no `arrivalDelay` at the closing stop for a large share of arrivals in some months (April, May, June, October: from ~30% to over 80%), and those missing delays are not random (the reported ones are skewed towards non-zero delays). Delay-based figures should not be trusted for these months. The column `N train arriving at terminus (delay reported)` is there for that: **compare it with `N train arriving at terminus`** before reading a punctuality share, and don't compute the share over the units with a reported delay only. SBB's data also nearly vanishes from 22 October to 1 November.
- **France**: from October 2025 the French agency also reports TER, so the mix of services in the French figures changes between September and October. Delays at French termini can be slightly understated (the agency snapshots the terminus a few minutes before the arrival).
- **Italy**: in daylight-saving months the actual times are one hour off the planned ones, but `arrivalDelay`, which is what is used here, is correct.
- **Buses and coaches** have no operator and are not rail; filter on `Route type` if needed.

The punctuality share at a threshold is `1 - N train delay (> X min) / N train arriving at terminus`, as in the operator summary.